# 《Improving and Generalizing Flow-Based Generative Models with Minibatch Optimal Transport》完整学习笔记

**论文：** *Improving and Generalizing Flow-Based Generative Models with Minibatch Optimal Transport*  
**作者：** Alexander Tong et al.  
**年份：** 2023  
**论文地址：** https://arxiv.org/abs/2302.00482

本文介绍 **Conditional Flow Matching（CFM）** 的广义框架，并在此基础上提出 Independent CFM、Optimal Transport CFM（OT-CFM）和 Schrödinger Bridge CFM（SB-CFM）。


---

# 第 0 章：论文总结与核心结论

## 0.1 论文要解决什么问题？

连续归一化流（Continuous Normalizing Flow，CNF）通过 ODE 将源分布 $q_0$ 转换为目标分布 $q_1$：

$$
\boxed{
\frac{dX_t}{dt}=v_\theta(t,X_t),
\qquad X_0\sim q_0
}
\tag{0.1}
$$

其中 $v_\theta$ 是神经网络表示的速度场。

传统 CNF 训练可能需要反复求解 ODE，因此计算成本较高。

Flow Matching 提供了另一种思路：**先构造一条已知的概率路径及其速度监督，再直接训练速度网络，而不是在训练中反复模拟 ODE。**

但是，已有的 Flow Matching 构造主要围绕高斯源分布展开，且不同端点之间的配对关系并没有被充分利用。

本文进一步提出：

> 将端点之间的联合分布（Coupling）作为概率路径设计的一部分，通过改变 Coupling 来控制整个概率流的运输性质。

其总体框架为：

$$
\boxed{
\begin{aligned}
&\text{给定源分布 }q_0\text{ 和目标分布 }q_1\\
&\Downarrow\\
&\text{选择端点耦合 }\pi(x_0,x_1)\\
&\Downarrow\\
&\text{构造条件概率路径 }p_t(x\mid x_0,x_1)\\
&\Downarrow\\
&\text{计算条件速度 }u_t(x\mid x_0,x_1)\\
&\Downarrow\\
&\text{通过 CFM 训练 }v_\theta(t,x)\\
&\Downarrow\\
&\text{推理时积分 ODE，得到目标分布}
\end{aligned}
}
\tag{0.2}
$$

## 0.2 本文最重要的四个结论

### 结论一：CFM 可以推广到任意可采样的源分布

原始 Flow Matching 的典型设置是：

$$
q_0=\mathcal N(0,I)
$$

本文允许：

$$
q_0
$$

是一般概率分布，只要能够从中采样。

通过构造条件概率路径，仍然可以使用简单的速度回归目标训练 CNF。

### 结论二：端点 Coupling 决定概率路径的运输性质

给定两个端点：

$$
(X_0,X_1)\sim\pi
$$

采用线性插值：

$$
X_t=(1-t)X_0+tX_1
$$

对应条件速度：

$$
U_t=X_1-X_0
$$

虽然插值公式相同，但如果选择不同的：

$$
\pi(x_0,x_1)
$$

就会产生不同的边缘概率路径。

因此，**CFM 不只是选择插值公式，还可以选择端点如何配对。**

### 结论三：真实 OT Coupling 与线性插值能够恢复 Dynamic OT

静态最优传输求解：

$$
\boxed{
\pi^*
=
\arg\min_{\pi\in\Pi(q_0,q_1)}
\mathbb E_\pi\|X_1-X_0\|^2
}
\tag{0.3}
$$

然后采用：

$$
\boxed{
X_t=(1-t)X_0+tX_1,
\qquad (X_0,X_1)\sim\pi^*
}
\tag{0.4}
$$

在适当数学条件下，该构造得到 Wasserstein 位移插值，也就是 Dynamic OT 的最优概率路径。

因此，OT-CFM 不需要直接优化整条 ODE 的动能，而是通过静态 OT 耦合构造具有最优运输性质的训练路径。

**但这里必须区分：**

- 真实全局 OT 耦合：具有严格的动态 OT 理论联系。
- Minibatch OT 耦合：是实际训练中的近似，不保证精确动态最优。
- 有限神经网络与有限训练：还会引入速度场拟合误差。

### 结论四：Entropic OT 与 Brownian Bridge 可以恢复 Schrödinger Bridge 的边缘概率流

Schrödinger Bridge（SB）寻找满足两个端点分布、同时尽可能接近参考随机过程的路径分布。

当参考过程为：

$$
dX_t=\sigma\,dW_t
$$

时，其最优端点耦合对应熵正则化 OT：

$$
\boxed{
\pi_{2\sigma^2}
=
\arg\min_{\pi\in\Pi(q_0,q_1)}
\left[
\mathbb E_\pi\|X_1-X_0\|^2
+
2\sigma^2D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)
\right]
}
\tag{0.5}
$$

给定端点后，使用 Brownian Bridge 条件路径：

$$
\boxed{
X_t=(1-t)X_0+tX_1
+\sigma\sqrt{t(1-t)}Z
}
\tag{0.6}
$$

其中：

$$
Z\sim\mathcal N(0,I)
$$

则 CFM 可以学习与 Schrödinger Bridge 具有相同单时刻边缘分布的确定性概率流。

这里的“相同”指：

$$
p_t^{\mathrm{CFM}}=p_t^{\mathrm{SB}}
$$

而不是确定性 ODE 的完整路径空间分布与随机 SB 完全相同。

## 0.3 三种主要 CFM 方法的区别

| 方法 | 端点耦合 | 条件概率路径 | 核心性质 |
|---|---|---|---|
| I-CFM | $\pi=q_0\otimes q_1$ | 线性均值 + 固定高斯噪声 | 不要求源分布为高斯 |
| OT-CFM | $\pi\approx\pi^*$ | 线性均值 + 固定高斯噪声 | 利用 OT 配对改善运输性质 |
| SB-CFM | $\pi=\pi_{2\sigma^2}$ | Brownian Bridge | 恢复 SB 的边缘概率路径 |

可以看到：

**I-CFM 与 OT-CFM 的主要区别在 Coupling；OT-CFM 与 SB-CFM 则不仅 Coupling 不同，条件概率路径也不同。**

---

# 第 1 章：Introduction

## 1.1 连续时间生成模型的基本目标

生成模型希望学习一个概率分布：

$$
q_1(x)
$$

例如真实图像数据分布。

由于直接从复杂数据分布采样比较困难，可以从简单的源分布开始：

$$
X_0\sim q_0
$$

然后通过某个可学习的变换：

$$
X_1=T_\theta(X_0)
$$

使：

$$
X_1\sim q_1
$$

连续时间生成模型将这个变换写成一个 ODE：

$$
\frac{dX_t}{dt}=v_\theta(t,X_t)
$$

其中：

$$
t\in[0,1]
$$

通过积分：

$$
X_1=X_0+\int_0^1v_\theta(t,X_t)\,dt
$$

完成分布转换。

因此，问题转化为：

$$
\boxed{
\text{如何学习一个速度场，使源分布演化为目标分布？}
}
$$

## 1.2 为什么传统 CNF 训练成本较高？

传统 CNF 可以通过最大似然等目标训练。

对于连续时间流，其对数密度满足：

$$
\boxed{
\frac{d}{dt}\log p_t(X_t)
=
-\nabla\cdot v_\theta(t,X_t)
}
\tag{1.1}
$$

因此：

$$
\log p_1(X_1)
=
\log p_0(X_0)
-
\int_0^1
\nabla\cdot v_\theta(t,X_t)\,dt
$$

训练时不仅需要求解状态 ODE，还需要处理散度积分及相应梯度计算。

这可能带来较高的计算开销。

Flow Matching 的关键变化是：

**不再要求训练过程中通过当前神经网络生成完整轨迹，而是直接监督速度场。**

例如，若能够构造：

$$
X_t
$$

及其目标速度：

$$
U_t
$$

则训练目标可以写为：

$$
\boxed{
\mathcal L(\theta)
=
\mathbb E\|v_\theta(t,X_t)-U_t\|^2
}
\tag{1.2}
$$

训练时只需要进行普通的网络前向传播和反向传播。

## 1.3 本文进一步提出的问题

虽然 Flow Matching 可以避免训练时的 ODE 模拟，但还存在一个重要设计自由度：

**训练样本中的起点和终点应该如何配对？**

假设：

$$
X_0\sim q_0,\qquad X_1\sim q_1
$$

最简单的方法是独立采样：

$$
\pi(x_0,x_1)=q_0(x_0)q_1(x_1)
$$

但这种随机配对可能导致大量不必要的长距离运输。

例如：

$$
q_0=\frac12\delta_0+\frac12\delta_{10}
$$

$$
q_1=\frac12\delta_1+\frac12\delta_{11}
$$

合理配对为：

$$
0\rightarrow1,\qquad10\rightarrow11
$$

而独立采样也可能产生：

$$
0\rightarrow11,\qquad10\rightarrow1
$$

后者的运输距离明显更大。

因此，本文提出：

$$
\boxed{
\text{通过优化端点 Coupling，改善整个生成概率流。}
}
$$

这就是 OT-CFM 的核心动机。

---

# 第 2 章：Preliminaries

本章建立理解全文所需要的两个数学框架：

1. **Neural ODE 与 Gaussian Probability Flow。**
2. **Static Optimal Transport 与 Dynamic Optimal Transport。**

## 2.1 Neural ODE 与 Continuous Normalizing Flow

### 2.1.1 什么是 Neural ODE？

普通 ODE 写成：

$$
\frac{dx(t)}{dt}=v(t,x(t))
$$

其中：

- $x(t)\in\mathbb R^d$：状态。
- $t$：时间。
- $v(t,x)$：速度场。

Neural ODE 使用神经网络表示速度场：

$$
\boxed{
\frac{dX_t}{dt}=v_\theta(t,X_t)
}
\tag{2.1}
$$

神经网络输入当前状态与时间，输出状态变化率。

给定初始状态：

$$
X_0=x_0
$$

通过积分得到：

$$
\boxed{
X_t=x_0+\int_0^t v_\theta(s,X_s)\,ds
}
\tag{2.2}
$$

所以 Neural ODE 的基本结构是：

$$
\boxed{
\text{神经网络预测速度}
+
\text{数值积分更新状态}
}
$$

### 2.1.2 从单个状态运动到概率分布演化

如果初始状态是随机变量：

$$
X_0\sim q_0
$$

那么每个样本都按照同一个速度场运动。

随着时间变化：

$$
X_t\sim p_t
$$

形成概率路径：

$$
\{p_t\}_{t\in[0,1]}
$$

其中：

$$
p_0=q_0
$$

我们希望：

$$
p_1=q_1
$$

这就是 CNF 的生成目标。

### 2.1.3 Probability Flow 与连续性方程

设：

$$
\frac{dX_t}{dt}=v_t(X_t)
$$

其中 $v_t$ 表示随时间变化的速度场。

由于概率质量在运动过程中守恒，概率密度满足：

$$
\boxed{
\partial_tp_t(x)
+
\nabla\cdot[p_t(x)v_t(x)]
=0
}
\tag{2.3}
$$

这就是连续性方程。

其中：

$$
p_t(x)v_t(x)
$$

表示概率流量。

它说明：

**某个区域的概率质量变化，来自概率质量通过边界的流入和流出。**

因此需要区分：

- Probability Path：$\{p_t\}$，描述每个时刻的概率分布。
- Velocity Field：$v_t(x)$，描述每个位置的运动速度。
- Probability Flow：速度场驱动的概率分布演化。

CFM 的任务，就是学习能够生成指定概率路径的速度场。

---

## 2.2 Gaussian Probability Flow

### 2.2.1 什么是 Gaussian Probability Path？

考虑一条概率路径：

$$
\boxed{
p_t(x)=\mathcal N(x;\mu_t,\sigma_t^2I)
}
\tag{2.4}
$$

其中：

- $\mu_t\in\mathbb R^d$：时刻 $t$ 的高斯均值。
- $\sigma_t>0$：时刻 $t$ 的高斯标准差。
- $I$：$d\times d$ 单位矩阵。

它表示每个时刻的概率分布都是高斯分布，但均值和方差随时间变化。

例如：

$$
\mu_t=5t,\qquad\sigma_t=1
$$

则：

$$
p_t=\mathcal N(5t,I)
$$

表示高斯分布从均值 $0$ 平移到均值 $5$。

如果：

$$
\mu_t=0,\qquad\sigma_t=1-t/2
$$

则高斯分布中心不变，但随时间逐渐收缩。

### 2.2.2 如何构造这条概率路径？

首先采样：

$$
Z\sim\mathcal N(0,I)
$$

定义：

$$
\boxed{
X_t=\mu_t+\sigma_tZ
}
\tag{2.5}
$$

由于高斯分布在线性变换下仍然是高斯：

$$
X_t\sim\mathcal N(\mu_t,\sigma_t^2I)
$$

因此式 (2.5) 就是这条概率路径的采样公式。

注意：

**这里的 $Z$ 对于一条具体轨迹是固定的随机变量，而不是每个时间重新采样的布朗噪声。**

因此式 (2.5) 定义的是一条可微的随机插值轨迹，而不是 SDE。

### 2.2.3 推导 Gaussian Probability Flow 的速度场

对：

$$
X_t=\mu_t+\sigma_tZ
$$

求时间导数：

$$
\dot X_t=\dot\mu_t+\dot\sigma_tZ
$$

由原式：

$$
Z=\frac{X_t-\mu_t}{\sigma_t}
$$

代入：

$$
\dot X_t
=
\dot\mu_t+
\frac{\dot\sigma_t}{\sigma_t}(X_t-\mu_t)
$$

因此得到速度场：

$$
\boxed{
u_t(x)=
\dot\mu_t+
\frac{\dot\sigma_t}{\sigma_t}(x-\mu_t)
}
\tag{2.6}
$$

这就是论文 Theorem 2.1 使用的 Gaussian Flow 速度公式。

其中：

$$
\dot\mu_t
$$

控制高斯均值的移动；

$$
\frac{\dot\sigma_t}{\sigma_t}(x-\mu_t)
$$

控制高斯分布的收缩与扩张。

当：

$$
\dot\sigma_t>0
$$

时，分布扩张。

当：

$$
\dot\sigma_t<0
$$

时，分布收缩。

**后面所有 Gaussian CFM 构造，本质上都是先指定 $\mu_t,\sigma_t$，再代入式 (2.6) 得到条件速度场。**

---

## 2.3 Static Optimal Transport

### 2.3.1 为什么需要 Optimal Transport？

假设：

$$
X_0\sim q_0,\qquad X_1\sim q_1
$$

我们希望将 $q_0$ 的概率质量运输到 $q_1$。

问题是：

**应该把哪个源样本运输到哪个目标样本？**

不同配对方式可能具有完全不同的运输成本。

Optimal Transport（OT）就是寻找总体运输成本最小的概率质量分配方案。

### 2.3.2 Coupling 是什么？

定义联合分布：

$$
\pi(x_0,x_1)
$$

它描述源样本 $x_0$ 与目标样本 $x_1$ 之间的配对关系。

要求：

$$
\int\pi(x_0,x_1)\,dx_1=q_0(x_0)
$$

$$
\int\pi(x_0,x_1)\,dx_0=q_1(x_1)
$$

也就是说，联合分布的两个边缘必须分别等于源分布和目标分布。

所有合法耦合构成集合：

$$
\boxed{
\Pi(q_0,q_1)
}
\tag{2.7}
$$

例如，独立耦合：

$$
\pi_{\mathrm{ind}}=q_0\otimes q_1
$$

一定是合法耦合。

但它通常不是运输成本最低的耦合。

### 2.3.3 Static OT 的优化目标

使用平方欧氏距离作为运输成本：

$$
c(x_0,x_1)=\|x_1-x_0\|^2
$$

定义：

$$
\boxed{
\mathcal C(\pi)
=
\mathbb E_{(X_0,X_1)\sim\pi}
\|X_1-X_0\|^2
}
\tag{2.8}
$$

Static OT 求解：

$$
\boxed{
\pi^*
=
\arg\min_{\pi\in\Pi(q_0,q_1)}
\mathcal C(\pi)
}
\tag{2.9}
$$

对应最优值：

$$
\boxed{
W_2^2(q_0,q_1)
=
\min_{\pi\in\Pi(q_0,q_1)}
\mathbb E_\pi\|X_1-X_0\|^2
}
\tag{2.10}
$$

其中 $W_2$ 是二阶 Wasserstein 距离。

称为 Static OT，是因为它只优化：

$$
(X_0,X_1)
$$

之间的端点耦合，不直接规定中间时间如何运动。

---

## 2.4 Dynamic Optimal Transport

### 2.4.1 Dynamic OT 优化什么？

Static OT 只关心：

$$
X_0\leftrightarrow X_1
$$

Dynamic OT 则考虑整个连续运输过程：

$$
X_0\rightarrow X_t\rightarrow X_1
$$

假设：

$$
\frac{dX_t}{dt}=v_t(X_t)
$$

并且：

$$
X_t\sim p_t
$$

定义整个过程的运输动能：

$$
\boxed{
\mathcal A(p,v)
=
\int_0^1
\mathbb E_{X_t\sim p_t}
\|v_t(X_t)\|^2dt
}
\tag{2.11}
$$

Dynamic OT 求解：

$$
\boxed{
\begin{aligned}
\min_{p,v}\quad&
\mathcal A(p,v)\\
\text{s.t.}\quad&
\partial_tp_t+\nabla\cdot(p_tv_t)=0\\
&p_0=q_0,\qquad p_1=q_1
\end{aligned}
}
\tag{2.12}
$$

因此：

- Static OT 优化端点联合分布 $\pi$。
- Dynamic OT 优化整条概率路径 $p_t$ 和速度场 $v_t$。

**Dynamic OT 不是要求每个中间时刻分别做一次最优配对，而是要求整个运输过程的总动能最小。**

### 2.4.2 为什么匀速直线是固定端点的最小动能路径？

考虑任意一条从 $x_0$ 到 $x_1$ 的可微轨迹：

$$
x(0)=x_0,\qquad x(1)=x_1
$$

由于：

$$
x_1-x_0=\int_0^1\dot x(t)\,dt
$$

根据 Cauchy–Schwarz 不等式：

$$
\left\|\int_0^1\dot x(t)\,dt\right\|^2
\le
\int_0^1\|\dot x(t)\|^2dt
$$

所以：

$$
\boxed{
\|x_1-x_0\|^2
\le
\int_0^1\|\dot x(t)\|^2dt
}
\tag{2.13}
$$

当速度恒定：

$$
\dot x(t)=x_1-x_0
$$

时等号成立。

积分得到：

$$
\boxed{
x(t)=(1-t)x_0+tx_1
}
\tag{2.14}
$$

因此：

$$
\boxed{
\text{固定端点之间的最小动能路径是匀速直线。}
}
$$

### 2.4.3 Benamou–Brenier 定理

Benamou–Brenier 定理建立了 Static OT 与 Dynamic OT 的联系：

$$
\boxed{
\underbrace{
\min_{\pi\in\Pi(q_0,q_1)}
\mathbb E_\pi\|X_1-X_0\|^2
}_{\text{Static OT}}
=
\underbrace{
\min_{p,v}
\int_0^1\mathbb E_{p_t}\|v_t(X_t)\|^2dt
}_{\text{Dynamic OT}}
}
\tag{2.15}
$$

右侧需要满足连续性方程和端点约束。

这两个问题的优化变量不同，但最优值相同。

下面推导为什么成立。

#### 第一步：任意动态运输的动能不低于 Static OT 最优成本

对于任意合法动态运输：

$$
\|X_1-X_0\|^2
\le
\int_0^1\|\dot X_t\|^2dt
$$

对所有轨迹取期望：

$$
\mathbb E\|X_1-X_0\|^2
\le
\mathbb E\int_0^1\|\dot X_t\|^2dt
$$

动态过程诱导出一个端点联合分布：

$$
\pi_{\mathrm{dyn}}=\operatorname{Law}(X_0,X_1)
$$

由于：

$$
\pi_{\mathrm{dyn}}\in\Pi(q_0,q_1)
$$

所以：

$$
W_2^2(q_0,q_1)
\le
\mathbb E_{\pi_{\mathrm{dyn}}}\|X_1-X_0\|^2
$$

结合得到：

$$
\boxed{
W_2^2(q_0,q_1)
\le
\mathcal A(p,v)
}
\tag{2.16}
$$

因此：

$$
\boxed{
\min_{p,v}\mathcal A(p,v)
\ge
W_2^2(q_0,q_1)
}
\tag{2.17}
$$

#### 第二步：使用真实 Static OT 耦合构造达到该成本的动态路径

选择：

$$
(X_0,X_1)\sim\pi^*
$$

其中：

$$
\pi^*
$$

是真实二次代价 OT 最优耦合。

构造：

$$
\boxed{
X_t=(1-t)X_0+tX_1
}
\tag{2.18}
$$

其轨迹速度为：

$$
\dot X_t=X_1-X_0
$$

于是：

$$
\begin{aligned}
\mathbb E_{\pi^*}
\int_0^1\|\dot X_t\|^2dt
&=
\mathbb E_{\pi^*}\|X_1-X_0\|^2\\
&=
W_2^2(q_0,q_1)
\end{aligned}
\tag{2.19}
$$

这里需要区分随机轨迹速度 $\dot X_t$ 与欧拉速度场 $v_t(x)$。

对于一般耦合，两者不能直接等同。

但真实二次代价 OT 耦合产生的位移插值具有特殊结构。在适当条件下，可以构造对应的欧拉速度场，使动态动能达到上述值。

例如，当最优耦合由 Monge Map $T$ 给出时：

$$
X_1=T(X_0)
$$

定义：

$$
T_t(x_0)=(1-t)x_0+tT(x_0)
$$

若 $T_t$ 在相关区域可逆，则：

$$
v_t(x)
=
T(T_t^{-1}(x))-T_t^{-1}(x)
$$

满足：

$$
v_t(X_t)=T(X_0)-X_0=\dot X_t
$$

因此构造出的合法动态路径达到：

$$
\boxed{
\mathcal A(p^*,v^*)=W_2^2(q_0,q_1)
}
\tag{2.20}
$$

所以：

$$
\boxed{
\min_{p,v}\mathcal A(p,v)
\le
W_2^2(q_0,q_1)
}
\tag{2.21}
$$

#### 第三步：合并两个不等式

由：

$$
\min_{p,v}\mathcal A(p,v)\ge W_2^2
$$

以及：

$$
\min_{p,v}\mathcal A(p,v)\le W_2^2
$$

最终：

$$
\boxed{
\min_{p,v}\mathcal A(p,v)
=
W_2^2(q_0,q_1)
}
\tag{2.22}
$$

因此：

$$
\boxed{
\text{真实 Static OT 耦合}
+
\text{匀速直线插值}
\Longrightarrow
\text{Dynamic OT 最优概率路径}
}
\tag{2.23}
$$

这个结论是 OT-CFM 的理论基础。

注意：这里的最优性来自**真实 OT 耦合与位移插值的构造**，并不是神经网络训练自动保证动态最优。

---

## 2.5 本章小结：为什么后面要使用静态 Coupling？

到这里，可以把两个数学框架连接起来。

CNF 的目标是学习：

$$
\frac{dX_t}{dt}=v_\theta(t,X_t)
$$

Dynamic OT 则进一步要求这条概率流的总动能尽可能小。

一种直接思路是在训练神经网络时加入：

$$
\int_0^1\mathbb E\|v_\theta(t,X_t)\|^2dt
$$

作为正则项，但计算它通常需要模拟当前网络生成的 ODE 轨迹。

本文采用另一种方法：

1. 在训练前或训练过程中，用 Static OT 确定端点之间的配对。
2. 对这些端点构造匀速直线插值。
3. 使用 CFM 回归这条概率路径的速度场。

这样，最优运输的结构被直接引入训练样本，而不是完全依赖神经网络在训练中自行发现。

这也解释了论文标题中的 **Minibatch Optimal Transport**：真实全局 OT 耦合通常不可直接采样，论文需要在有限 Batch 中近似计算它。

---

# 第 3 章：Conditional Flow Matching — ODEs from Static Couplings

本章是论文的核心，依次介绍广义 CFM、I-CFM、OT-CFM 和 SB-CFM。

## 3.1 Vector Fields Generating Mixtures of Probability Paths

### 3.1.1 为什么需要广义 CFM？

前面假设我们知道一条概率路径：

$$
p_t(x)
$$

及其对应速度场：

$$
u_t(x)
$$

那么可以直接使用 Flow Matching：

$$
\boxed{
\mathcal L_{\mathrm{FM}}(\theta)
=
\mathbb E_{t,X_t\sim p_t}
\|v_\theta(t,X_t)-u_t(X_t)\|^2
}
\tag{3.1}
$$

但实际问题是：

虽然我们可能知道如何构造条件概率路径：

$$
p_t(x\mid z)
$$

却通常无法直接计算边缘速度场：

$$
u_t(x)
$$

因为它涉及对条件变量 $z$ 的积分。

CFM 的思路是：

**不直接计算边缘速度场，而是使用容易计算的条件速度场作为随机监督目标。**

### 3.1.2 条件概率路径与边缘概率路径

定义条件变量：

$$
Z\sim q(z)
$$

给定 $Z=z$，构造条件概率路径：

$$
p_t(x\mid z)
$$

对应条件速度场：

$$
u_t(x\mid z)
$$

边缘概率路径为：

$$
\boxed{
p_t(x)=\int p_t(x\mid z)q(z)\,dz
}
\tag{3.2}
$$

这里：

- $q(z)$：条件变量的采样分布。
- $p_t(x\mid z)$：给定条件后，时刻 $t$ 的概率密度。
- $u_t(x\mid z)$：生成该条件概率路径的速度场。
- $p_t(x)$：对所有条件路径混合得到的边缘分布。

### 3.1.3 Theorem 3.1：条件路径混合对应的边缘速度场

论文证明，在适当正则性条件下，边缘速度场可以写成：

$$
\boxed{
u_t(x)
=
\frac{
\int u_t(x\mid z)p_t(x\mid z)q(z)\,dz
}{
p_t(x)
}
}
\tag{3.3}
$$

等价于：

$$
u_t(x)=\mathbb E[u_t(X_t\mid Z)\mid X_t=x]
$$

这个定理的推导已经熟悉，这里不再重复。

需要记住的结论是：

**只要条件概率路径及其速度场合法，就能够通过混合得到对应的边缘概率路径与速度场。**

### 3.1.4 Theorem 3.2：CFM 与 FM 的目标等价

定义 CFM 损失：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}(\theta)
=
\mathbb E_{t,Z,X_t\mid Z}
\|v_\theta(t,X_t)-u_t(X_t\mid Z)\|^2
}
\tag{3.4}
$$

论文证明，在其定理条件及相关可积性条件下：

$$
\boxed{
\mathcal L_{\mathrm{CFM}}(\theta)
=
\mathcal L_{\mathrm{FM}}(\theta)+C
}
\tag{3.5}
$$

其中 $C$ 与 $\theta$ 无关。

因此：

$$
\boxed{
\nabla_\theta\mathcal L_{\mathrm{CFM}}
=
\nabla_\theta\mathcal L_{\mathrm{FM}}
}
\tag{3.6}
$$

也就是说，两种目标具有相同的参数梯度。

CFM 的优势在于：它不需要显式计算通常难以获得的边缘速度场。

### 3.1.5 Algorithm 1：通用 CFM 训练算法

给定三个对象：

$$
q(z),\qquad p_t(x\mid z),\qquad u_t(x\mid z)
$$

训练时执行：

1. 采样条件 $z\sim q(z)$。
2. 采样时间 $t\sim\mathcal U(0,1)$。
3. 采样状态 $x\sim p_t(x\mid z)$。
4. 计算目标速度 $u_t(x\mid z)$。
5. 使用 MSE 更新网络：

$$
\boxed{
\ell(\theta)
=
\|v_\theta(t,x)-u_t(x\mid z)\|^2
}
\tag{3.7}
$$

因此，广义 CFM 的关键不在于某一种固定插值，而在于如何选择这三个对象。

后面的几种方法正是对它们作不同选择。

---

## 3.2 Sources of Conditional Probability Paths

### 3.2.1 FM from the Gaussian

论文首先说明，已有的 Gaussian Flow Matching 可以写成广义 CFM 的一个特例。

#### 条件变量

选择：

$$
z=x_1,\qquad x_1\sim q_1
$$

也就是只对目标样本进行条件化。

#### 条件概率路径

给定目标样本 $x_1$，定义：

$$
\boxed{
p_t(x\mid x_1)
=
\mathcal N\left(
x;
tx_1,
[1-(1-\sigma)t]^2I
\right)
}
\tag{3.8}
$$

其中 $\sigma>0$ 是终点平滑标准差。

这里的均值与标准差分别为：

$$
\mu_t=tx_1
$$

$$
\sigma_t=1-(1-\sigma)t
$$

因此可以直接采样：

$$
\boxed{
X_t=tx_1+[1-(1-\sigma)t]Z,
\quad Z\sim\mathcal N(0,I)
}
\tag{3.9}
$$

#### 条件速度场

使用第 2 章推导的 Gaussian Flow 公式：

$$
u_t(x\mid x_1)
=
\dot\mu_t+
\frac{\dot\sigma_t}{\sigma_t}(x-\mu_t)
$$

由于：

$$
\dot\mu_t=x_1
$$

$$
\dot\sigma_t=-(1-\sigma)
$$

得到：

$$
u_t(x\mid x_1)
=
x_1-
\frac{1-\sigma}{1-(1-\sigma)t}(x-tx_1)
$$

通分整理：

$$
\boxed{
u_t(x\mid x_1)
=
\frac{x_1-(1-\sigma)x}
{1-(1-\sigma)t}
}
\tag{3.10}
$$

这就是对应的条件速度监督。

#### 为什么这种方法要求高斯源分布？

观察起点：

$$
t=0
$$

此时：

$$
X_0=Z
$$

因此：

$$
p_0=\mathcal N(0,I)
$$

而在终点：

$$
t=1
$$

有：

$$
X_1^{\mathrm{path}}=x_1+\sigma Z
$$

因此，边缘终点分布为：

$$
p_1=q_1*\mathcal N(0,\sigma^2I)
$$

其中 $*$ 表示卷积。

也就是说，该构造天然从标准高斯出发，终点是经过高斯平滑的数据分布。

**本文接下来要做的第一件事，就是去掉这个高斯源分布限制。**

---

## 3.2.2 Basic Form of CFM：Independent Coupling

### 3.2.2.1 从单端点条件扩展到双端点条件

前面只使用目标样本：

$$
z=x_1
$$

现在改为：

$$
\boxed{
z=(x_0,x_1)
}
\tag{3.11}
$$

其中：

$$
x_0\sim q_0,\qquad x_1\sim q_1
$$

I-CFM 使用独立耦合：

$$
\boxed{
q(z)=q_0(x_0)q_1(x_1)
}
\tag{3.12}
$$

这意味着：

$$
X_0\perp X_1
$$

源样本和目标样本独立采样。

这里 $q_0$ 不需要是高斯分布。

### 3.2.2.2 构造条件 Gaussian Path

给定端点：

$$
(x_0,x_1)
$$

定义：

$$
\boxed{
p_t(x\mid x_0,x_1)
=
\mathcal N\left(
x;
(1-t)x_0+tx_1,
\sigma^2I
\right)
}
\tag{3.13}
$$

这表示：

- 高斯均值从 $x_0$ 线性移动到 $x_1$。
- 高斯标准差始终保持为 $\sigma$。

对应采样公式：

$$
\boxed{
X_t=(1-t)x_0+tx_1+\sigma Z
}
\tag{3.14}
$$

其中：

$$
Z\sim\mathcal N(0,I)
$$

注意：

$$
(1-t)x_0+tx_1
$$

是均值的线性插值；

$$
\sigma Z
$$

是固定尺度的高斯平滑。

### 3.2.2.3 推导条件速度场

由：

$$
\mu_t=(1-t)x_0+tx_1
$$

得到：

$$
\dot\mu_t=x_1-x_0
$$

又因为：

$$
\sigma_t=\sigma
$$

所以：

$$
\dot\sigma_t=0
$$

代入 Gaussian Flow 公式：

$$
u_t(x\mid x_0,x_1)
=
\dot\mu_t+
\frac{\dot\sigma_t}{\sigma_t}(x-\mu_t)
$$

得到：

$$
\boxed{
u_t(x\mid x_0,x_1)=x_1-x_0
}
\tag{3.15}
$$

因此，条件速度是一个常量。

这意味着：

**给定一对端点之后，I-CFM 的条件 ODE 速度不依赖当前位置和时间。**

### 3.2.2.4 I-CFM 训练目标

采样：

$$
X_0\sim q_0
$$

$$
X_1\sim q_1
$$

$$
t\sim\mathcal U(0,1)
$$

$$
Z\sim\mathcal N(0,I)
$$

构造：

$$
X_t=(1-t)X_0+tX_1+\sigma Z
$$

目标速度：

$$
U=X_1-X_0
$$

最终：

$$
\boxed{
\mathcal L_{\mathrm{I-CFM}}
=
\mathbb E
\left[
\left\|
v_\theta(t,X_t)-(X_1-X_0)
\right\|^2
\right]
}
\tag{3.16}
$$

训练不需要数值积分 ODE。

### 3.2.2.5 Proposition 3.3：端点分布

当：

$$
t=0
$$

时：

$$
X_t=X_0+\sigma Z
$$

因此：

$$
\boxed{
p_0=q_0*\mathcal N(0,\sigma^2I)
}
\tag{3.17}
$$

当：

$$
t=1
$$

时：

$$
X_t=X_1+\sigma Z
$$

因此：

$$
\boxed{
p_1=q_1*\mathcal N(0,\sigma^2I)
}
\tag{3.18}
$$

当：

$$
\sigma\to0
$$

时：

$$
p_0\to q_0,\qquad p_1\to q_1
$$

因此，I-CFM 可以处理一般源分布，但在有限 $\sigma>0$ 时，严格对应的是经过高斯平滑的端点分布。

### 3.2.2.6 I-CFM 的问题是什么？

虽然条件速度：

$$
X_1-X_0
$$

非常简单，但独立耦合并不考虑运输距离。

例如：

$$
q_0=\frac12\delta_0+\frac12\delta_{10}
$$

$$
q_1=\frac12\delta_1+\frac12\delta_{11}
$$

独立耦合会以相同概率考虑四种端点组合：

$$
(0,1),\quad(0,11),\quad(10,1),\quad(10,11)
$$

其中：

$$
0\rightarrow11
$$

和：

$$
10\rightarrow1
$$

显然是不必要的长距离运输。

因此，即使每条条件路径都是直线，整体运输也不一定最优。

这引出了 OT-CFM。

---

## 3.2.3 Optimal Transport CFM

### 3.2.3.1 OT-CFM 的核心变化

I-CFM 使用：

$$
\pi_{\mathrm{ind}}=q_0\otimes q_1
$$

OT-CFM 改为：

$$
\boxed{
(X_0,X_1)\sim\pi^*
}
\tag{3.19}
$$

其中：

$$
\boxed{
\pi^*
=
\arg\min_{\pi\in\Pi(q_0,q_1)}
\mathbb E_\pi\|X_1-X_0\|^2
}
\tag{3.20}
$$

其他部分保持不变：

$$
\boxed{
X_t=(1-t)X_0+tX_1+\sigma Z
}
\tag{3.21}
$$

$$
\boxed{
U=X_1-X_0
}
\tag{3.22}
$$

所以：

$$
\boxed{
\mathcal L_{\mathrm{OT-CFM}}
=
\mathbb E_{\substack{
(X_0,X_1)\sim\pi^*\\
t,Z
}}
\left[
\|v_\theta(t,X_t)-(X_1-X_0)\|^2
\right]
}
\tag{3.23}
$$

**OT-CFM 与 I-CFM 的主要区别不是插值公式，也不是损失函数形式，而是端点联合分布。**

### 3.2.3.2 Proposition 3.4：为什么 OT-CFM 与 Dynamic OT 有关？

根据第 2 章：

$$
\min_{\pi\in\Pi(q_0,q_1)}
\mathbb E_\pi\|X_1-X_0\|^2
=
\min_{p,v}\mathcal A(p,v)
$$

如果使用真实 OT 耦合：

$$
(X_0,X_1)\sim\pi^*
$$

并令：

$$
\sigma=0
$$

则：

$$
X_t=(1-t)X_0+tX_1
$$

构成 Wasserstein 位移插值。

因此，所构造的边缘概率路径就是动态最优传输路径。

需要区分三个层次：

**第一层：条件路径。**

每一对端点之间都是匀速直线：

$$
X_t=(1-t)X_0+tX_1
$$

**第二层：边缘概率路径。**

所有条件路径混合得到：

$$
p_t(x)
$$

当端点耦合是真实二次代价 OT 耦合时，它构成动态 OT 的最优概率路径。

**第三层：神经网络生成路径。**

训练得到：

$$
v_\theta(t,x)
$$

并积分：

$$
\frac{dY_t}{dt}=v_\theta(t,Y_t)
$$

只有在网络充分准确地拟合对应速度场时，生成路径才会接近理论上的动态 OT。

所以：

$$
\boxed{
\text{真实 OT Coupling}
+
\text{零噪声线性插值}
\Longrightarrow
\text{Dynamic OT}
}
\tag{3.24}
$$

但：

$$
\boxed{
\text{Minibatch OT}
+
\text{有限噪声}
+
\text{有限网络}
}
$$

只能近似这一性质。

### 3.2.3.3 为什么需要 Minibatch OT？

真实 OT 耦合：

$$
\pi^*
$$

通常无法直接采样。

假设训练数据包含：

$$
N
$$

个源样本和：

$$
N
$$

个目标样本。

如果在整个数据集上构造离散 OT，需要计算：

$$
N\times N
$$

的运输成本矩阵。

当 $N$ 很大时，存储和计算开销都很高。

因此，论文使用 Minibatch OT：

**每次只从两个分布中采样一个 Batch，在 Batch 内计算近似最优配对。**

---

### 3.2.3.4 Minibatch OT：从样本到运输计划矩阵

假设 Batch Size 为：

$$
B
$$

从源分布采样：

$$
\mathcal B_0=
\{x_0^{(1)},\ldots,x_0^{(B)}\}
$$

从目标分布采样：

$$
\mathcal B_1=
\{x_1^{(1)},\ldots,x_1^{(B)}\}
$$

这里的上标：

$$
(i)
$$

表示 Batch 中第 $i$ 个样本。

#### 第一步：构造成本矩阵 $C$

定义：

$$
\boxed{
C_{ij}=\|x_0^{(i)}-x_1^{(j)}\|^2
}
\tag{3.25}
$$

因此：

$$
C\in\mathbb R^{B\times B}
$$

矩阵的行表示源样本，列表示目标样本。

例如：

$$
\mathcal B_0=\{0,10\}
$$

$$
\mathcal B_1=\{1,11\}
$$

则：

$$
C=
\begin{bmatrix}
(0-1)^2&(0-11)^2\\
(10-1)^2&(10-11)^2
\end{bmatrix}
$$

得到：

$$
\boxed{
C=
\begin{bmatrix}
1&121\\
81&1
\end{bmatrix}
}
\tag{3.26}
$$

这个矩阵表示不同配对方式的运输成本。

但它还没有决定具体应该运输多少概率质量。

#### 第二步：定义运输计划矩阵 $P$

定义：

$$
\boxed{
P\in\mathbb R_+^{B\times B}
}
\tag{3.27}
$$

其中：

$$
P_{ij}\ge0
$$

表示从第 $i$ 个源样本运输到第 $j$ 个目标样本的概率质量。

由于 Batch 中的样本被赋予相同权重：

$$
\frac1B
$$

所以每个源样本需要运出：

$$
\frac1B
$$

的概率质量。

每个目标样本也需要接收：

$$
\frac1B
$$

的概率质量。

因此：

$$
\boxed{
\sum_{j=1}^BP_{ij}=\frac1B
}
\tag{3.28}
$$

表示每一行的和等于 $1/B$。

同时：

$$
\boxed{
\sum_{i=1}^BP_{ij}=\frac1B
}
\tag{3.29}
$$

表示每一列的和等于 $1/B$。

#### 第三步：将行列约束写成矩阵形式

定义：

$$
\mathbf1=
\begin{bmatrix}
1\\
\vdots\\
1
\end{bmatrix}
\in\mathbb R^B
$$

那么：

$$
P\mathbf1
$$

就是对 $P$ 的每一行求和。

因此：

$$
\boxed{
P\mathbf1=\frac1B\mathbf1
}
\tag{3.30}
$$

表示每行总运输质量为 $1/B$。

类似地：

$$
P^\top\mathbf1
$$

表示对 $P$ 的每一列求和。

因此：

$$
\boxed{
P^\top\mathbf1=\frac1B\mathbf1
}
\tag{3.31}
$$

表示每列总接收质量为 $1/B$。

这两个约束保证了概率质量守恒。

#### 第四步：最小化总运输成本

由于：

$$
P_{ij}
$$

表示运输的概率质量；

$$
C_{ij}
$$

表示单位概率质量的运输成本；

所以总成本为：

$$
\boxed{
\sum_{i=1}^B\sum_{j=1}^BP_{ij}C_{ij}
}
\tag{3.32}
$$

因此离散 OT 问题为：

$$
\boxed{
\begin{aligned}
P^*=\arg\min_{P\ge0}\quad&
\sum_{i,j}P_{ij}C_{ij}\\
\text{s.t.}\quad&
P\mathbf1=\frac1B\mathbf1\\
&P^\top\mathbf1=\frac1B\mathbf1
\end{aligned}
}
\tag{3.33}
$$

对于前面的例子：

$$
C=
\begin{bmatrix}
1&121\\
81&1
\end{bmatrix}
$$

最优运输计划为：

$$
\boxed{
P^*=
\begin{bmatrix}
0.5&0\\
0&0.5
\end{bmatrix}
}
\tag{3.34}
$$

它表示：

$$
0\rightarrow1
$$

$$
10\rightarrow11
$$

各运输 $0.5$ 的概率质量。

总成本为：

$$
0.5\times1+0.5\times1=1
$$

如果采用交叉配对：

$$
P=
\begin{bmatrix}
0&0.5\\
0.5&0
\end{bmatrix}
$$

则总成本为：

$$
0.5\times121+0.5\times81=101
$$

因此前一种配对明显更优。

#### 第五步：根据 $P^*$ 生成端点配对

求得：

$$
P^*
$$

之后，从其表示的离散联合分布中采样：

$$
(I,J)\sim P^*
$$

得到：

$$
X_0=x_0^{(I)}
$$

$$
X_1=x_1^{(J)}
$$

然后构造：

$$
X_t=(1-t)X_0+tX_1+\sigma Z
$$

目标速度：

$$
U=X_1-X_0
$$

即可进行 CFM 训练。

对于等权、等样本数、无熵正则化的离散 OT，存在一个最优运输计划是缩放后的置换矩阵，因此也可以把它理解为 Batch 内的最优重新配对。

但一般的 OT 运输计划并不一定必须是确定性的一一配对。

---

### 3.2.3.5 Minibatch OT 为什么不会破坏边缘分布？

这是论文一个重要性质。

对于固定 Batch，运输计划满足：

$$
\sum_jP^*_{ij}=\frac1B
$$

因此：

$$
\mathbb P(I=i\mid\mathcal B_0,\mathcal B_1)=\frac1B
$$

也就是说，按照运输计划采样后，每个源样本仍然以相同概率被选中。

因此，源端点的条件边缘分布仍是：

$$
\widehat q_0^{(B)}
=
\frac1B\sum_{i=1}^B\delta_{x_0^{(i)}}
$$

对随机 Batch 取期望：

$$
\mathbb E_{\mathcal B_0}
\widehat q_0^{(B)}=q_0
$$

同理：

$$
\mathbb E_{\mathcal B_1}
\widehat q_1^{(B)}=q_1
$$

所以 Minibatch OT 在所有随机 Batch 上诱导出的平均耦合：

$$
\overline\pi_B
$$

仍满足：

$$
\boxed{
\overline\pi_B\in\Pi(q_0,q_1)
}
\tag{3.35}
$$

因此：

**Minibatch OT 改变的是端点配对关系，而不是源分布和目标分布本身。**

但一般有：

$$
\boxed{
\overline\pi_B\ne\pi^*
}
\tag{3.36}
$$

所以它保留了正确的端点边缘分布，却不保证达到全局 OT 最优。

---

## 3.2.4 Schrödinger Bridge CFM

这一节的目标是进一步推广 OT-CFM：不再只考虑确定性的最小成本运输，而是考虑具有随机性的最优运输过程。

理解 SB-CFM 需要依次建立四个概念：

1. Schrödinger Bridge 优化的是什么。
2. Entropic OT 为什么出现。
3. Brownian Bridge 是什么。
4. 为什么 Brownian Bridge 的概率路径可以用确定性 ODE 生成。

### 3.2.4.1 Schrödinger Bridge 的基本问题

先考虑一个参考随机过程：

$$
\boxed{
dX_t=\sigma\,dW_t
}
\tag{3.37}
$$

其中：

- $W_t$：标准 Brownian Motion。
- $\sigma>0$：扩散系数。
- $X_0\sim q_0$：初始分布。

该过程没有漂移，只有随机扩散。

设其完整路径分布为：

$$
\mathbb R
$$

这里的 $\mathbb R$ 不是某个时刻的状态分布，而是整个随机过程：

$$
(X_t)_{t\in[0,1]}
$$

在路径空间上的概率分布。

现在希望寻找另一个随机过程，其路径分布为：

$$
\mathbb P
$$

满足：

$$
P_0=q_0
$$

$$
P_1=q_1
$$

并尽可能接近参考过程 $\mathbb R$。

于是定义 Schrödinger Bridge：

$$
\boxed{
\mathbb P^*
=
\arg\min_{\substack{
\mathbb P:\,P_0=q_0\\
P_1=q_1
}}
D_{\mathrm{KL}}(\mathbb P\|\mathbb R)
}
\tag{3.38}
$$

这里：

$$
D_{\mathrm{KL}}(\mathbb P\|\mathbb R)
$$

衡量两个完整随机过程的路径分布之间的差异。

因此：

**Schrödinger Bridge 不是单纯寻找距离最短的确定性运输，而是寻找满足端点约束、同时尽可能保留参考随机过程特征的随机运输。**

### 3.2.4.2 什么是 Entropic Optimal Transport？

普通 Static OT 求解：

$$
\boxed{
\min_{\pi\in\Pi(q_0,q_1)}
\mathbb E_\pi\|X_1-X_0\|^2
}
\tag{3.39}
$$

它只考虑运输成本。

Entropic OT 在运输成本之外增加熵正则项：

$$
\boxed{
\min_{\pi\in\Pi(q_0,q_1)}
\left[
\mathbb E_\pi\|X_1-X_0\|^2
-\varepsilon H(\pi)
\right]
}
\tag{3.40}
$$

其中：

- $H(\pi)$：联合分布的熵。
- $\varepsilon>0$：熵正则化强度。

对于离散联合分布 $P$：

$$
\boxed{
H(P)=-\sum_{i,j}P_{ij}\log P_{ij}
}
\tag{3.41}
$$

熵越大，运输计划通常越分散。

因此，式 (3.40) 包含两个相互竞争的目标：

$$
\mathbb E_\pi\|X_1-X_0\|^2
$$

希望运输距离尽可能小；

$$
-\varepsilon H(\pi)
$$

鼓励运输计划具有一定随机性，而不是全部概率质量都集中在少数配对上。

#### 为什么熵正则化 OT 也可以写成 KL 形式？

论文使用：

$$
\boxed{
\min_{\pi\in\Pi(q_0,q_1)}
\left[
\mathbb E_\pi\|X_1-X_0\|^2
+
\varepsilon D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)
\right]
}
\tag{3.42}
$$

其中：

$$
q_0\otimes q_1
$$

表示独立耦合。

对于离散分布，令：

$$
q_0(i)=a_i,\qquad q_1(j)=b_j
$$

以及：

$$
\pi(i,j)=P_{ij}
$$

则：

$$
D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)
=
\sum_{i,j}P_{ij}\log\frac{P_{ij}}{a_ib_j}
$$

展开对数：

$$
\begin{aligned}
D_{\mathrm{KL}}
={}&
\sum_{i,j}P_{ij}\log P_{ij}\\
&-\sum_{i,j}P_{ij}\log a_i\\
&-\sum_{i,j}P_{ij}\log b_j
\end{aligned}
\tag{3.43}
$$

第一项：

$$
\sum_{i,j}P_{ij}\log P_{ij}=-H(\pi)
$$

第二项利用行和约束：

$$
\sum_jP_{ij}=a_i
$$

得到：

$$
\begin{aligned}
\sum_{i,j}P_{ij}\log a_i
&=
\sum_i\left(\sum_jP_{ij}\right)\log a_i\\
&=
\sum_i a_i\log a_i\\
&=-H(q_0)
\end{aligned}
$$

第三项利用列和约束：

$$
\sum_iP_{ij}=b_j
$$

得到：

$$
\sum_{i,j}P_{ij}\log b_j=-H(q_1)
$$

因此：

$$
\boxed{
D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)
=
-H(\pi)+H(q_0)+H(q_1)
}
\tag{3.44}
$$

由于 OT 优化过程中：

$$
q_0,\qquad q_1
$$

固定，所以：

$$
H(q_0)+H(q_1)
$$

与优化变量 $\pi$ 无关。

将式 (3.44) 代入式 (3.42)：

$$
\begin{aligned}
\mathcal J(\pi)
={}&
\mathbb E_\pi\|X_1-X_0\|^2
-\varepsilon H(\pi)\\
&+\varepsilon H(q_0)+\varepsilon H(q_1)
\end{aligned}
$$

最后两项是常数，因此：

$$
\boxed{
\begin{aligned}
&\arg\min_{\pi\in\Pi(q_0,q_1)}
\left[
\mathbb E_\pi[c]
+\varepsilon D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)
\right]\\
&\qquad=
\arg\min_{\pi\in\Pi(q_0,q_1)}
\left[
\mathbb E_\pi[c]-\varepsilon H(\pi)
\right]
\end{aligned}
}
\tag{3.45}
$$

这里的等价指**最优解相同**，不是两个目标函数的数值完全相同。

连续分布也有相应恒等式，但需要 KL 与微分熵表达有定义；使用 KL 形式通常更稳妥。

#### 熵正则化系数的作用

当：

$$
\varepsilon\to0
$$

时，运输成本占主导，最优计划趋向普通 OT 的最优解集合。

当：

$$
\varepsilon\to\infty
$$

时，KL 正则项占主导，最优耦合趋向：

$$
q_0\otimes q_1
$$

即独立耦合。

因此：

$$
\boxed{
\text{小 }\varepsilon
\rightarrow
\text{更接近确定性最优配对}
}
$$

$$
\boxed{
\text{大 }\varepsilon
\rightarrow
\text{更接近独立随机配对}
}
$$

---

### 3.2.4.3 为什么 Schrödinger Bridge 对应 Entropic OT？

现在回到参考布朗运动：

$$
dX_t=\sigma\,dW_t
$$

给定：

$$
X_0=x_0
$$

有：

$$
X_1=x_0+\sigma W_1
$$

由于：

$$
W_1\sim\mathcal N(0,I)
$$

所以：

$$
\boxed{
X_1\mid X_0=x_0
\sim
\mathcal N(x_0,\sigma^2I)
}
\tag{3.46}
$$

因此，参考过程的端点联合分布为：

$$
R_{01}(dx_0,dx_1)
=
q_0(dx_0)
\mathcal N(x_1;x_0,\sigma^2I)\,dx_1
$$

其中高斯转移密度为：

$$
\mathcal N(x_1;x_0,\sigma^2I)
\propto
\exp\left(
-\frac{\|x_1-x_0\|^2}{2\sigma^2}
\right)
$$

Schrödinger Bridge 的路径空间 KL 优化可以通过 KL 的链式分解，先分解为端点耦合的优化和给定端点后的条件路径优化。

对于固定端点耦合 $\pi$，最优条件路径仍采用参考布朗过程的条件路径，即 Brownian Bridge。

因此，端点耦合需要最小化：

$$
D_{\mathrm{KL}}(\pi\|R_{01})
$$

下面将它展开。

由定义：

$$
D_{\mathrm{KL}}(\pi\|R_{01})
=
\mathbb E_\pi
\log
\frac{\pi(X_0,X_1)}
{q_0(X_0)r(X_1\mid X_0)}
$$

其中：

$$
r(x_1\mid x_0)
=
\mathcal N(x_1;x_0,\sigma^2I)
$$

在分子分母中引入 $q_1(X_1)$：

$$
\begin{aligned}
D_{\mathrm{KL}}(\pi\|R_{01})
={}&
\mathbb E_\pi
\log
\frac{\pi(X_0,X_1)}
{q_0(X_0)q_1(X_1)}\\
&+
\mathbb E_\pi
\log
\frac{q_1(X_1)}
{r(X_1\mid X_0)}
\end{aligned}
\tag{3.47}
$$

第一项为：

$$
D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)
$$

第二项中：

$$
-\log r(x_1\mid x_0)
=
\frac{\|x_1-x_0\|^2}{2\sigma^2}
+\frac d2\log(2\pi\sigma^2)
$$

而：

$$
\mathbb E_\pi\log q_1(X_1)
$$

只取决于固定目标边缘 $q_1$。

因此：

$$
\boxed{
\begin{aligned}
D_{\mathrm{KL}}(\pi\|R_{01})
={}&
D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)\\
&+
\frac1{2\sigma^2}
\mathbb E_\pi\|X_1-X_0\|^2
+C
\end{aligned}
}
\tag{3.48}
$$

其中 $C$ 与 $\pi$ 无关。

两边乘以：

$$
2\sigma^2
$$

得到：

$$
\boxed{
\begin{aligned}
2\sigma^2D_{\mathrm{KL}}(\pi\|R_{01})
={}&
\mathbb E_\pi\|X_1-X_0\|^2\\
&+
2\sigma^2D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)
+C'
\end{aligned}
}
\tag{3.49}
$$

因此，最优端点耦合为：

$$
\boxed{
\pi_{2\sigma^2}
=
\arg\min_{\pi\in\Pi(q_0,q_1)}
\left[
\mathbb E_\pi\|X_1-X_0\|^2
+
2\sigma^2D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)
\right]
}
\tag{3.50}
$$

这就解释了论文为什么使用：

$$
\boxed{
\varepsilon=2\sigma^2
}
$$

**这个系数不是任意选择的，而是由参考布朗运动的扩散强度推导出来的。**

---

### 3.2.4.4 Brownian Motion 与 Brownian Bridge 有什么区别？

这是理解 SB-CFM 的关键。

#### Brownian Motion：无条件随机扩散

普通布朗运动满足：

$$
\boxed{
dX_t=\sigma\,dW_t
}
\tag{3.51}
$$

给定初始位置：

$$
X_0=x_0
$$

其解为：

$$
X_t=x_0+\sigma W_t
$$

因此：

$$
\boxed{
X_t\sim\mathcal N(x_0,\sigma^2tI)
}
\tag{3.52}
$$

均值：

$$
\mathbb E[X_t]=x_0
$$

保持不变。

方差：

$$
\operatorname{Var}(X_t)=\sigma^2tI
$$

随时间增加。

因此普通 Brownian Motion 没有漂移，只有扩散。

#### Brownian Bridge：固定起点和终点的条件布朗运动

现在要求：

$$
X_0=x_0
$$

并且：

$$
X_1=x_1
$$

也就是考虑：

**所有从 $x_0$ 出发，并在 $t=1$ 到达 $x_1$ 的布朗运动路径。**

这就是 Brownian Bridge。

它不是无条件 Brownian Motion，而是 Brownian Motion 在固定终点条件下的条件过程。

因为终点必须到达 $x_1$，所以该过程自然具有向终点靠近的趋势。

### 3.2.4.5 Brownian Bridge 的条件分布推导

普通布朗运动满足：

$$
X_t=x_0+\sigma W_t
$$

$$
X_1=x_0+\sigma W_1
$$

已知：

$$
\operatorname{Var}(W_t)=tI
$$

$$
\operatorname{Var}(W_1)=I
$$

以及：

$$
\operatorname{Cov}(W_t,W_1)=tI
$$

由于：

$$
(W_t,W_1)
$$

是联合高斯随机变量，可以使用高斯条件分布公式。

条件均值为：

$$
\mathbb E[W_t\mid W_1]=tW_1
$$

而：

$$
W_1=\frac{x_1-x_0}{\sigma}
$$

所以：

$$
\begin{aligned}
\mathbb E[X_t\mid X_0=x_0,X_1=x_1]
&=
x_0+\sigma tW_1\\
&=
x_0+t(x_1-x_0)\\
&=
(1-t)x_0+tx_1
\end{aligned}
$$

因此：

$$
\boxed{
\mu_t=(1-t)x_0+tx_1
}
\tag{3.53}
$$

条件协方差为：

$$
\begin{aligned}
\operatorname{Var}(W_t\mid W_1)
&=
\operatorname{Var}(W_t)
-\operatorname{Cov}(W_t,W_1)
\operatorname{Var}(W_1)^{-1}
\operatorname{Cov}(W_1,W_t)\\
&=
tI-tI\cdot I^{-1}\cdot tI\\
&=
t(1-t)I
\end{aligned}
$$

因此：

$$
\boxed{
\operatorname{Var}(X_t\mid X_0,X_1)
=
\sigma^2t(1-t)I
}
\tag{3.54}
$$

最终：

$$
\boxed{
p_t(x\mid x_0,x_1)
=
\mathcal N\left(
x;
(1-t)x_0+tx_1,
\sigma^2t(1-t)I
\right)
}
\tag{3.55}
$$

这就是论文使用的 Brownian Bridge 条件概率路径。

#### 为什么方差在两个端点都是零？

观察：

$$
\sigma^2t(1-t)
$$

当：

$$
t=0
$$

时，方差为零，因为起点已经固定。

当：

$$
t=1
$$

时，方差也为零，因为终点已经固定。

在：

$$
t=\frac12
$$

时，方差达到最大值：

$$
\frac{\sigma^2}{4}
$$

因此，Brownian Bridge 的随机性表现为：

$$
\boxed{
\text{起点确定}
\rightarrow
\text{中间随机波动}
\rightarrow
\text{终点确定}
}
$$

这与普通布朗运动的方差随时间单调增长不同。

---

### 3.2.4.6 Brownian Bridge 为什么具有漂移？

普通 Brownian Motion 满足：

$$
dX_t=\sigma\,dW_t
$$

但给定终点 $X_1=x_1$ 后，Brownian Bridge 的 SDE 变为：

$$
\boxed{
dX_t=
\frac{x_1-X_t}{1-t}\,dt
+
\sigma\,dW_t
}
\tag{3.56}
$$

其中：

$$
0\le t<1
$$

这里出现了漂移：

$$
\boxed{
b(t,x)=\frac{x_1-x}{1-t}
}
\tag{3.57}
$$

其含义是：

- 如果当前位置距离目标很远，漂移会推动状态向终点移动。
- 随着 $t\to1$，剩余时间减少，漂移强度增大。
- 该条件过程最终到达指定终点。

因此：

**Brownian Bridge 的漂移来自对终点的条件化，而不是在原始 Brownian Motion 中人为添加了漂移。**

更严格地说，这个条件化过程可以通过 Doob $h$-transform 推导，详细证明放在附录。

---

### 3.2.4.7 SB-CFM 的条件 ODE 速度场

现在已经得到 Brownian Bridge 的条件概率路径：

$$
p_t(x\mid x_0,x_1)
=
\mathcal N(x;\mu_t,s_t^2I)
$$

其中：

$$
\mu_t=(1-t)x_0+tx_1
$$

$$
s_t=\sigma\sqrt{t(1-t)}
$$

注意这里：

- $\sigma$：参考布朗运动的扩散系数。
- $s_t$：时刻 $t$ 的条件高斯标准差。

为了避免混淆，用两个不同符号表示。

根据 Gaussian Probability Flow：

$$
u_t(x\mid x_0,x_1)
=
\dot\mu_t+
\frac{\dot s_t}{s_t}(x-\mu_t)
$$

首先：

$$
\dot\mu_t=x_1-x_0
$$

其次：

$$
s_t=\sigma[t(1-t)]^{1/2}
$$

求导：

$$
\dot s_t
=
\frac{\sigma(1-2t)}
{2\sqrt{t(1-t)}}
$$

因此：

$$
\frac{\dot s_t}{s_t}
=
\frac{1-2t}{2t(1-t)}
$$

最终：

$$
\boxed{
\begin{aligned}
u_t(x\mid x_0,x_1)
={}&x_1-x_0\\
&+
\frac{1-2t}{2t(1-t)}
\left[
x-(1-t)x_0-tx_1
\right]
\end{aligned}
}
\tag{3.58}
$$

这就是 SB-CFM 使用的条件 ODE 速度场。

对应采样公式为：

$$
\boxed{
X_t=(1-t)X_0+tX_1+\sigma\sqrt{t(1-t)}Z
}
\tag{3.59}
$$

其中：

$$
Z\sim\mathcal N(0,I)
$$

代入：

$$
X_t-\mu_t=\sigma\sqrt{t(1-t)}Z
$$

还可以写成：

$$
\boxed{
u_t(X_t\mid X_0,X_1)
=
X_1-X_0+
\frac{\sigma(1-2t)}
{2\sqrt{t(1-t)}}Z
}
\tag{3.60}
$$

需要注意，当：

$$
t\to0
$$

或：

$$
t\to1
$$

时，第二项的系数可能发散。

因此，实际训练中需要处理端点附近的数值稳定性。

---

### 3.2.4.8 为什么 SB-CFM 的 ODE 速度不等于 Brownian Bridge 的 SDE 漂移？

现在有两个公式。

Brownian Bridge 的 SDE 漂移：

$$
\boxed{
b(t,x)=\frac{x_1-x}{1-t}
}
\tag{3.61}
$$

SB-CFM 的条件 ODE 速度：

$$
\boxed{
u_t(x)
=
x_1-x_0+
\frac{1-2t}{2t(1-t)}(x-\mu_t)
}
\tag{3.62}
$$

它们为什么不同？

因为 SDE 的概率分布变化由漂移和扩散共同决定，而 ODE 没有扩散项。

对于：

$$
dX_t=b(t,X_t)dt+\sigma dW_t
$$

对应 Fokker–Planck 方程：

$$
\boxed{
\partial_tp_t
=
-\nabla\cdot(bp_t)
+
\frac{\sigma^2}{2}\Delta p_t
}
\tag{3.63}
$$

而确定性 ODE：

$$
dX_t=u_t(X_t)dt
$$

对应连续性方程：

$$
\boxed{
\partial_tp_t=-\nabla\cdot(u_tp_t)
}
\tag{3.64}
$$

为了让二者具有相同的边缘概率路径，可以选择：

$$
\boxed{
u_t(x)
=
b(t,x)
-
\frac{\sigma^2}{2}\nabla\log p_t(x)
}
\tag{3.65}
$$

对于 Brownian Bridge：

$$
p_t(x\mid x_0,x_1)
=
\mathcal N(x;\mu_t,\sigma^2t(1-t)I)
$$

其 Score 为：

$$
\boxed{
\nabla_x\log p_t(x\mid x_0,x_1)
=
-\frac{x-\mu_t}{\sigma^2t(1-t)}
}
\tag{3.66}
$$

代入式 (3.65)：

$$
u_t(x)
=
\frac{x_1-x}{1-t}
+
\frac{x-\mu_t}{2t(1-t)}
$$

利用：

$$
x_1-x=(1-t)(x_1-x_0)-(x-\mu_t)
$$

整理得到：

$$
\boxed{
u_t(x)
=
x_1-x_0+
\frac{1-2t}{2t(1-t)}(x-\mu_t)
}
\tag{3.67}
$$

与式 (3.58) 完全一致。

因此：

$$
\boxed{
\text{Brownian Bridge SDE 漂移}
\ne
\text{SB-CFM 条件 ODE 速度}
}
$$

两者的关系是：

$$
\boxed{
\text{ODE 速度}
=
\text{SDE 漂移}
-
\frac{\sigma^2}{2}\text{Score}
}
$$

这解释了为什么一个带随机扩散的 Brownian Bridge，可以对应一条具有相同单时刻边缘分布的确定性概率流。

---

### 3.2.4.9 Proposition 3.5：SB-CFM 恢复 Schrödinger Bridge 的边缘概率流

现在把前面的内容组合起来。

首先，从熵正则化 OT 中采样端点：

$$
\boxed{
(X_0,X_1)\sim\pi_{2\sigma^2}
}
\tag{3.68}
$$

然后，给定端点，使用 Brownian Bridge：

$$
\boxed{
p_t(x\mid X_0,X_1)
=
\mathcal N\left(
x;
(1-t)X_0+tX_1,
\sigma^2t(1-t)I
\right)
}
\tag{3.69}
$$

最后，将所有条件路径混合：

$$
\boxed{
p_t(x)
=
\int
p_t(x\mid x_0,x_1)
\,d\pi_{2\sigma^2}(x_0,x_1)
}
\tag{3.70}
$$

根据 Schrödinger Bridge 的端点分解性质，这条边缘概率路径与参考布朗过程对应的最优 Schrödinger Bridge 的单时刻边缘一致。

因此，CFM 学习到的 ODE 可以生成相同的边缘概率路径。

$$
\boxed{
p_t^{\mathrm{SB-CFM}}
=
p_t^{\mathrm{SB}}
}
\tag{3.71}
$$

但需要强调：

$$
\boxed{
\text{边缘概率路径一致}
\ne
\text{完整随机路径分布一致}
}
$$

SB-CFM 使用确定性 ODE；原始 Schrödinger Bridge 是随机过程。

两者可以具有相同的每时刻概率分布，却具有不同的路径随机性。

---

### 3.2.4.10 SB-CFM 的训练流程

SB-CFM 可以概括为：

**第一步：** 从源分布和目标分布中采样 Batch。

**第二步：** 求解熵正则化 OT，获得近似端点耦合：

$$
P_\varepsilon
$$

其中对于指定参考布朗过程：

$$
\varepsilon=2\sigma^2
$$

**第三步：** 根据运输计划采样端点对：

$$
(X_0,X_1)
$$

**第四步：** 采样：

$$
t\sim\mathcal U(0,1)
$$

$$
Z\sim\mathcal N(0,I)
$$

**第五步：** 构造 Brownian Bridge 状态：

$$
X_t=(1-t)X_0+tX_1+\sigma\sqrt{t(1-t)}Z
$$

**第六步：** 计算条件 ODE 速度：

$$
U_t=
X_1-X_0+
\frac{\sigma(1-2t)}
{2\sqrt{t(1-t)}}Z
$$

**第七步：** 使用 MSE 训练：

$$
\boxed{
\mathcal L_{\mathrm{SB-CFM}}
=
\mathbb E\|v_\theta(t,X_t)-U_t\|^2
}
\tag{3.72}
$$

整个训练过程不需要模拟 Schrödinger Bridge 的随机轨迹，也不需要在训练中积分生成 ODE。

---

# 第 4 章：Related Work

原论文第 4 章主要讨论已有方法与本文的关系。按照你的要求，这里不展开其他论文，只保留理解本文创新点所需要的内容。

## 4.1 Simulation-Free Continuous-Time Modeling

传统 CNF 训练往往需要在训练过程中求解当前模型定义的 ODE。

CFM 则预先构造：

$$
X_t\sim p_t(x\mid z)
$$

及其监督速度：

$$
u_t(X_t\mid z)
$$

然后训练：

$$
\mathcal L_{\mathrm{CFM}}
=
\mathbb E\|v_\theta(t,X_t)-u_t(X_t\mid z)\|^2
$$

因此：

$$
\boxed{
\text{CFM 的 Simulation-Free 指训练时不需要模拟生成 ODE。}
}
$$

推理时仍然需要求解：

$$
\frac{dX_t}{dt}=v_\theta(t,X_t)
$$

本文的进一步贡献，是将这种训练方式推广到一般源分布和不同端点 Coupling。

## 4.2 Dynamic Optimal Transport

已有 Neural OT 方法可以直接对 CNF 的动能进行正则化：

$$
\mathcal R(\theta)
=
\int_0^1
\mathbb E\|v_\theta(t,X_t)\|^2dt
$$

但计算该目标通常需要沿着当前网络生成的轨迹积分。

OT-CFM 则通过：

$$
\pi^*
$$

和线性插值：

$$
X_t=(1-t)X_0+tX_1
$$

在构造训练路径时引入 OT 结构。

所以：

$$
\boxed{
\text{直接优化动态动能}
\quad\longrightarrow\quad
\text{通过静态 OT 构造具有最优性质的训练路径}
}
$$

这是本文在训练方法上的重要区别。[arXiv](https://arxiv.org/html/2302.00482?utm_source=chatgpt.com)

---

# 第 5 章：Experiments

实验部分主要验证三个问题：

1. OT-CFM 是否能够学习更接近 Dynamic OT 的概率流？
2. OT-CFM 是否能够提高训练和推理效率？
3. CFM 是否能够处理一般源分布及不同生成任务？

## 5.1 Low-Dimensional Data：Optimal Transport and Faster Convergence

### 5.1.1 实验设置

论文考虑四组二维分布运输任务：

| 源分布 | 目标分布 |
|---|---|
| Standard Gaussian | 8 Gaussians |
| Moons | 8 Gaussians |
| Standard Gaussian | Moons |
| Standard Gaussian | S-Curve |

其中：

$$
\text{Moons}\rightarrow\text{8 Gaussians}
$$

尤其重要，因为源分布不是高斯分布。

论文比较 OT-CFM、I-CFM、FM 及其他 Neural OT 方法。

### 5.1.2 如何评价是否接近 Dynamic OT？

根据第 2 章，Dynamic OT 的最小动能为：

$$
W_2^2(q_0,q_1)
$$

因此可以计算模型实际生成路径的动能：

$$
\boxed{
\operatorname{PE}(v_\theta)
=
\mathbb E_{X_0\sim q_0}
\int_0^1
\|v_\theta(t,X_t)\|^2dt
}
\tag{5.1}
$$

其中 PE 表示 Path Energy。

然后定义 Normalized Path Energy：

$$
\boxed{
\operatorname{NPE}(v_\theta)
=
\frac{
\left|
\operatorname{PE}(v_\theta)
-
W_2^2(q_0,q_1)
\right|
}{
W_2^2(q_0,q_1)
}
}
\tag{5.2}
$$

NPE 越小，说明实际路径动能越接近理论最优值。

但需要注意：

**NPE 小不等价于生成分布一定正确。**

如果模型没有准确运输到目标分布，仅仅动能接近某个数值并不足以证明它实现了 Dynamic OT。

因此论文同时报告生成分布的拟合误差。

### 5.1.3 实验结果

原论文 Table 2 的部分 NPE 结果如下：

| 方法 | Gaussian → 8 Gaussians | Moons → 8 Gaussians | Gaussian → Moons | Gaussian → S-Curve |
|---|---:|---:|---:|---:|
| OT-CFM | **0.018** | **0.053** | 0.087 | **0.027** |
| I-CFM | 0.222 | 2.738 | 0.841 | 0.867 |
| 2-RF | 0.069 | 0.149 | **0.076** | 0.112 |
| 3-RF | 0.055 | 0.123 | 0.084 | 0.129 |

表中为五次随机种子的均值，省略标准差。

可以看到，OT-CFM 在四组任务中的三组取得了最低 NPE。

特别是：

$$
\text{Moons}\rightarrow\text{8 Gaussians}
$$

任务中：

$$
\operatorname{NPE}_{\mathrm{I-CFM}}=2.738
$$

而：

$$
\operatorname{NPE}_{\mathrm{OT-CFM}}=0.053
$$

说明端点 Coupling 的选择对运输动能具有显著影响。

论文还发现，OT-CFM 通常具有更好的训练收敛表现，并且在相同 NFE 下获得更低的生成误差。

这里 NFE 表示：

$$
\boxed{
\text{Number of Function Evaluations}
}
$$

即 ODE 求解器调用速度网络的次数。

这说明 OT-CFM 学到的速度场在实验中更容易被数值积分。

但应注意，动态动能较低并不在数学上直接保证任意 ODE 求解器的误差较低。[arXiv](https://arxiv.org/html/2302.00482?utm_source=chatgpt.com)

### 5.1.4 SB-CFM 实验

论文还比较了 SB-CFM 与 Diffusion Schrödinger Bridge（DSB）。

| 任务 | SB-CFM | DSB |
|---|---:|---:|
| Gaussian → 8 Gaussians | **0.454** | 1.440 |
| Moons → 8 Gaussians | **1.377** | 2.407 |
| Gaussian → Moons | **0.283** | 0.333 |
| Gaussian → S-Curve | **0.297** | 0.383 |

该指标是学习到的中间边缘分布与参考 Schrödinger Bridge 边缘分布之间的平均 $W_2$ 误差。

SB-CFM 在这些实验中取得了更低的误差，支持其恢复 SB 边缘概率流的理论结论。[arXiv](https://arxiv.org/html/2302.00482?utm_source=chatgpt.com)

---

## 5.2 Application to Single-Cell Interpolation

### 5.2.1 任务是什么？

假设在不同时间点观察到细胞状态分布：

$$
q_{t_0},q_{t_1},\ldots,q_{t_n}
$$

但不知道每个细胞在时间之间的真实运动轨迹。

希望学习一个连续时间动力系统：

$$
\frac{dX_t}{dt}=v_\theta(t,X_t)
$$

能够描述细胞群体状态随时间的变化。

论文采用留一时间点验证：训练时移除一个中间时间点的数据，再使用学习到的动力学预测该时间点的细胞分布。

### 5.2.2 为什么适合 CFM？

这里的源分布和目标分布通常不是高斯分布。

而且数据只提供不同时间点的样本集合，没有真实的样本级配对。

OT-CFM 可以利用两个时间点之间的样本分布构造 Minibatch OT 耦合，再训练连续时间速度场。

### 5.2.3 实验结果

原论文 Table 4：

| 方法 | CITE-seq | EB | Multiome |
|---|---:|---:|---:|
| DSB | 0.953 | 0.862 | 1.079 |
| I-CFM | 0.965 | 0.872 | 1.085 |
| SB-CFM | 1.067 | 1.221 | 1.129 |
| OT-CFM | **0.882** | **0.790** | **0.937** |

评价指标为留出时间点的 $W_1$ 距离，越低越好。

OT-CFM 在三个数据集上都取得了最低平均误差。

这说明，在这些数据集上，使用 OT 耦合学习分布演化比独立耦合更有效。

同时，SB-CFM 并没有取得最优结果，说明具有 Schrödinger Bridge 理论性质，并不意味着在所有实际分布插值任务上都更准确。[arXiv](https://arxiv.org/html/2302.00482?utm_source=chatgpt.com)

---

## 5.3 High-Dimensional Data：Lower-Cost Training and Inference

### 5.3.1 实验设置

论文使用 CIFAR-10 进行无条件图像生成。

源分布为：

$$
q_0=\mathcal N(0,I)
$$

目标分布为真实图像数据分布。

速度网络使用时间条件 U-Net。

论文还调整了训练配置，例如：

- 学习率 $2\times10^{-4}$。
- 梯度裁剪范数为 $1$。
- EMA 衰减系数为 $0.9999$。
- 基础通道数为 $128$。
- Dropout 为 $10\%$。
- Batch Size 为 $128$。

### 5.3.2 评价指标

使用 FID：

$$
\operatorname{FID}
$$

衡量生成图像与真实图像在特征空间中的统计差异。

FID 越低，通常表示生成质量越好。

论文还比较不同 NFE 下的生成效果。

### 5.3.3 实验结果

原论文 Table 5 的部分结果：

| 方法 | Euler 100 NFE | Euler 1000 NFE | Adaptive |
|---|---:|---:|---:|
| VP-FM | 7.772 | 4.048 | 4.335 |
| OT-FM | 4.640 | 3.822 | 3.655 |
| I-CFM | 4.461 | **3.643** | 3.659 |
| OT-CFM | **4.443** | 3.741 | **3.577** |

可以看到：

在 100 NFE 时，OT-CFM 的 FID 最低。

在 1000 NFE 时，I-CFM 略优于 OT-CFM。

在自适应 ODE 求解器下，OT-CFM 取得表中最低的 FID。

因此，实验更准确的结论是：

$$
\boxed{
\text{OT-CFM 在较低 NFE 下通常具有较好的生成效率。}
}
$$

而不是 OT-CFM 在所有采样步数下都绝对优于其他方法。

论文还报告，在其图像实验配置中，Minibatch OT 带来的额外训练计算开销低于约 $1\%$。这一数字取决于网络规模、Batch Size 和 OT 求解实现。[arXiv](https://arxiv.org/html/2302.00482?utm_source=chatgpt.com)

---

## 5.4 OT-CFM for Unsupervised Translation

### 5.4.1 任务定义

考虑两个没有样本级配对关系的数据分布：

$$
q_0,\qquad q_1
$$

希望学习：

$$
q_0\rightarrow q_1
$$

例如：

$$
\text{不微笑的人脸}
\rightarrow
\text{微笑的人脸}
$$

训练数据中并没有同一个人的微笑与不微笑图像成对出现。

### 5.4.2 实验设置

论文使用 CelebA 数据集。

首先训练 VAE，将图像编码为 128 维潜在向量。

然后在潜在空间中使用 CFM 学习不同属性分布之间的运输。

这样可以降低直接在高维图像空间求解运输问题的计算成本。

### 5.4.3 实验结果

原论文 Table 6：

| 方法 | $\sigma=0.1$ | $\sigma=0.3$ | $\sigma=1$ |
|---|---:|---:|---:|
| Identity | 9.17 | 9.17 | 9.17 |
| I-CFM | 4.85 | 3.44 | 1.59 |
| OT-CFM | **2.81** | **1.91** | **1.04** |

评价指标为 MMD，单位为 $10^{-3}$，越低越好。

OT-CFM 在三个平滑尺度下都取得了更低的 MMD。

这说明 Minibatch OT 耦合也能够改善无配对数据分布转换任务。[arXiv](https://arxiv.org/html/2302.00482?utm_source=chatgpt.com)

---

## 5.5 Additional Experiments and Extensions

论文还研究了三个扩展方向。

### 5.5.1 OT-CFM 的监督方差

CFM 使用的监督目标：

$$
U=X_1-X_0
$$

是随机变量。

即使给定：

$$
X_t=x
$$

也可能存在多组端点：

$$
(X_0,X_1)
$$

产生同一个中间状态，却对应不同的监督速度。

因此：

$$
\operatorname{Var}(U\mid X_t=x)
$$

可能不为零。

OT Coupling 能够在某些条件下减少这种不确定性。

尤其在存在适当 Monge Map、且 $\sigma\to0$ 的理想情形下，中间状态可以唯一确定对应的运输速度，监督目标的条件方差可以趋于零。

这为 OT-CFM 训练更稳定提供了理论解释。

### 5.5.2 Energy-Based CFM

论文还考虑只有未归一化密度：

$$
R_0(x),\qquad R_1(x)
$$

而无法直接从目标分布采样的情况。

通过重要性采样，可以构造加权 CFM 目标。

其主要推导放在附录。

### 5.5.3 Stochastic Dynamics

论文讨论了将 CFM 推广到随机动力学的方向。

这一部分不是本文的核心理论结果，因此这里不进一步展开。

---

# 第 6 章：Conclusion

## 6.1 论文的主要贡献

本文提出广义 Conditional Flow Matching 框架，使连续时间生成模型能够在不进行训练期 ODE 模拟的情况下，学习一般源分布到目标分布的概率流。

其中最重要的贡献是：

**第一，推广了 Flow Matching 的适用范围。**

不再要求源分布一定是标准高斯，也不需要显式计算源分布密度。

**第二，引入端点 Coupling 作为概率路径的设计自由度。**

同样的线性插值公式，在不同端点 Coupling 下可以产生不同的边缘概率路径。

**第三，通过 OT-CFM 建立 Static OT 与 Dynamic OT 的联系。**

真实 OT 耦合与线性位移插值能够构造动态最优传输路径。

Minibatch OT 则提供了实际可计算的近似。

**第四，通过 SB-CFM 恢复 Schrödinger Bridge 的边缘概率流。**

使用熵正则化 OT 端点耦合和 Brownian Bridge 条件概率路径，可以通过确定性 ODE 学习对应的边缘分布演化。

## 6.2 论文的局限

原论文明确指出两个主要局限。

### 局限一：需要可计算的条件概率路径与速度场

CFM 的核心要求是：

$$
p_t(x\mid z)
$$

可以采样，并且：

$$
u_t(x\mid z)
$$

可以计算。

如果希望直接对复杂的边缘速度场施加约束，就可能难以继续保持简单的 Simulation-Free 训练形式。

### 局限二：Minibatch OT 在高维空间中存在近似误差

Minibatch OT 只在有限样本上求解运输计划。

因此：

$$
\overline\pi_B
$$

一般不等于真实全局 OT 耦合：

$$
\pi^*
$$

在高维空间中，这种近似误差可能更加明显。

所以 Minibatch OT-CFM 不保证精确恢复 Dynamic OT。[arXiv](https://arxiv.org/html/2302.00482?utm_source=chatgpt.com)

---

# 附录 A：Monge Map、Kantorovich Coupling 与 Dynamic OT

## A.1 Monge Map 是什么？

Static OT 有两种常见表述。

第一种是 Monge 形式。

寻找确定性映射：

$$
T:\mathbb R^d\rightarrow\mathbb R^d
$$

满足：

$$
T_\#q_0=q_1
$$

其中：

$$
T_\#q_0
$$

表示将 $X_0\sim q_0$ 通过 $T$ 变换后得到的分布。

优化目标为：

$$
\boxed{
T^*
=
\arg\min_{T:\,T_\#q_0=q_1}
\mathbb E_{X_0\sim q_0}
\|T(X_0)-X_0\|^2
}
\tag{A.1}
$$

这里每个源样本都被映射到一个确定的目标位置。

第二种是 Kantorovich 形式。

不直接优化确定性映射，而是优化联合分布：

$$
\boxed{
\pi^*
=
\arg\min_{\pi\in\Pi(q_0,q_1)}
\mathbb E_\pi\|X_1-X_0\|^2
}
\tag{A.2}
$$

Kantorovich 形式允许一个源位置的概率质量被分配到多个目标位置，因此更一般。

当最优 Monge Map 存在时，它诱导一个最优耦合：

$$
\boxed{
\pi^*=(\mathrm{Id},T^*)_\#q_0
}
\tag{A.3}
$$

其中：

$$
\mathrm{Id}(x)=x
$$

表示恒等映射。

## A.2 从 Monge Map 构造 Dynamic OT

假设存在最优 Monge Map：

$$
T^*
$$

定义：

$$
\boxed{
T_t(x_0)=(1-t)x_0+tT^*(x_0)
}
\tag{A.4}
$$

于是：

$$
X_t=T_t(X_0)
$$

并且：

$$
\dot X_t=T^*(X_0)-X_0
$$

在 $T_t$ 可逆的区域：

$$
X_0=T_t^{-1}(X_t)
$$

因此欧拉速度场为：

$$
\boxed{
v_t(x)
=
T^*(T_t^{-1}(x))-T_t^{-1}(x)
}
\tag{A.5}
$$

沿着实际轨迹：

$$
v_t(X_t)=T^*(X_0)-X_0
$$

所以：

$$
\begin{aligned}
\mathcal A(p,v)
&=
\mathbb E\int_0^1\|v_t(X_t)\|^2dt\\
&=
\mathbb E\|T^*(X_0)-X_0\|^2\\
&=
W_2^2(q_0,q_1)
\end{aligned}
$$

因此：

$$
\boxed{
\mathcal A(p,v)=W_2^2(q_0,q_1)
}
\tag{A.6}
$$

这就是从最优 Monge Map 构造动态最优运输的直接方式。

---

# 附录 B：为什么 Minibatch OT 保持端点边缘分布？

设两个随机 Batch：

$$
\mathcal B_0=
\{X_0^{(1)},\ldots,X_0^{(B)}\}
$$

$$
\mathcal B_1=
\{X_1^{(1)},\ldots,X_1^{(B)}\}
$$

分别从：

$$
q_0,\qquad q_1
$$

独立采样。

求解离散 OT：

$$
P^*(\mathcal B_0,\mathcal B_1)
$$

并根据：

$$
\mathbb P(I=i,J=j\mid\mathcal B_0,\mathcal B_1)=P^*_{ij}
$$

采样端点索引。

由于：

$$
\sum_jP^*_{ij}=\frac1B
$$

所以：

$$
\mathbb P(I=i\mid\mathcal B_0,\mathcal B_1)=\frac1B
$$

因此，对于任意可测集合 $A$：

$$
\begin{aligned}
\mathbb P(X_0^{(I)}\in A)
&=
\mathbb E_{\mathcal B_0,\mathcal B_1}
\left[
\sum_i\frac1B
\mathbf1_{\{X_0^{(i)}\in A\}}
\right]\\
&=
\frac1B\sum_i
\mathbb P(X_0^{(i)}\in A)\\
&=
q_0(A)
\end{aligned}
$$

同理：

$$
\mathbb P(X_1^{(J)}\in A)=q_1(A)
$$

所以：

$$
\boxed{
\overline\pi_B\in\Pi(q_0,q_1)
}
\tag{B.1}
$$

但这个证明并没有得到：

$$
\overline\pi_B=\pi^*
$$

因此：

$$
\boxed{
\text{保持边缘分布正确}
\ne
\text{达到全局 OT 最优}
}
$$

---

# 附录 C：Brownian Bridge 的漂移推导

## C.1 普通布朗运动的转移密度

考虑：

$$
dX_t=\sigma\,dW_t
$$

从时刻 $t$ 的位置 $x$ 出发，在时刻 $1$ 到达 $x_1$ 的转移密度为：

$$
\boxed{
h(t,x)
=
\mathcal N(x_1;x,\sigma^2(1-t)I)
}
\tag{C.1}
$$

展开：

$$
h(t,x)
\propto
\exp\left(
-\frac{\|x_1-x\|^2}{2\sigma^2(1-t)}
\right)
$$

因此：

$$
\log h(t,x)
=
-\frac{\|x_1-x\|^2}{2\sigma^2(1-t)}
+C(t)
$$

其中 $C(t)$ 与 $x$ 无关。

对 $x$ 求梯度：

$$
\boxed{
\nabla_x\log h(t,x)
=
\frac{x_1-x}{\sigma^2(1-t)}
}
\tag{C.2}
$$

## C.2 Doob $h$-transform

对于扩散过程：

$$
dX_t=b(t,X_t)dt+\sigma dW_t
$$

其生成元为：

$$
\mathcal Lf
=
b\cdot\nabla f
+
\frac{\sigma^2}{2}\Delta f
$$

Doob $h$-transform 对应的条件化漂移为：

$$
\boxed{
b^h(t,x)
=
b(t,x)+\sigma^2\nabla_x\log h(t,x)
}
\tag{C.3}
$$

对于普通布朗运动：

$$
b(t,x)=0
$$

因此：

$$
b^h(t,x)
=
\sigma^2
\frac{x_1-x}{\sigma^2(1-t)}
$$

得到：

$$
\boxed{
b^h(t,x)=\frac{x_1-x}{1-t}
}
\tag{C.4}
$$

所以 Brownian Bridge 满足：

$$
\boxed{
dX_t=
\frac{x_1-X_t}{1-t}dt+\sigma dW_t
}
\tag{C.5}
$$

这说明 Brownian Bridge 的漂移来自对终点的条件化。

---

# 附录 D：为什么 SB-CFM 的 ODE 与 Brownian Bridge SDE 具有相同边缘分布？

考虑 SDE：

$$
dX_t=b(t,X_t)dt+\sigma dW_t
$$

其概率密度满足：

$$
\partial_tp_t
=
-\nabla\cdot(bp_t)
+
\frac{\sigma^2}{2}\Delta p_t
$$

利用：

$$
\Delta p_t
=
\nabla\cdot(p_t\nabla\log p_t)
$$

得到：

$$
\begin{aligned}
\partial_tp_t
&=
-\nabla\cdot(bp_t)
+
\nabla\cdot
\left(
\frac{\sigma^2}{2}p_t\nabla\log p_t
\right)\\
&=
-\nabla\cdot
\left[
\left(
b-\frac{\sigma^2}{2}\nabla\log p_t
\right)p_t
\right]
\end{aligned}
$$

因此可以定义确定性 ODE：

$$
\boxed{
\frac{dX_t}{dt}
=
u_t(X_t)
}
$$

其中：

$$
\boxed{
u_t(x)
=
b(t,x)-\frac{\sigma^2}{2}\nabla\log p_t(x)
}
\tag{D.1}
$$

该 ODE 与原 SDE 满足相同的边缘概率演化方程。

对于 Brownian Bridge：

$$
b(t,x)=\frac{x_1-x}{1-t}
$$

以及：

$$
p_t(x\mid x_0,x_1)
=
\mathcal N(x;\mu_t,\sigma^2t(1-t)I)
$$

有：

$$
\nabla_x\log p_t(x\mid x_0,x_1)
=
-\frac{x-\mu_t}{\sigma^2t(1-t)}
$$

代入：

$$
u_t(x)
=
\frac{x_1-x}{1-t}
+
\frac{x-\mu_t}{2t(1-t)}
$$

整理：

$$
\boxed{
u_t(x)
=
x_1-x_0+
\frac{1-2t}{2t(1-t)}(x-\mu_t)
}
\tag{D.2}
$$

这与 SB-CFM 的 Gaussian Flow 速度公式一致。

因此，SB-CFM 的条件 ODE 不是随意设计的，而是能够产生 Brownian Bridge 相同条件边缘分布的一种确定性概率流。

---

# 附录 E：条件速度监督的方差与 OT-CFM

设：

$$
U=X_1-X_0
$$

以及：

$$
X_t=(1-t)X_0+tX_1+\sigma Z
$$

CFM 训练使用：

$$
U
$$

作为监督目标。

但在给定：

$$
X_t=x
$$

之后，端点对：

$$
(X_0,X_1)
$$

通常仍然具有随机性。

因此：

$$
U
$$

也可能具有条件方差：

$$
\operatorname{Var}(U\mid X_t=x)
$$

考虑：

$$
\boxed{
\mathbb E
\left[
\|U-\mathbb E[U\mid X_t]\|^2
\right]
}
\tag{E.1}
$$

该量衡量在已知模型输入 $X_t$ 后，监督目标仍然具有多少无法消除的随机性。

如果使用独立耦合，不同端点对可能经过相同或相近的中间位置，但对应不同速度。

例如：

$$
0\rightarrow10
$$

和：

$$
10\rightarrow0
$$

在：

$$
t=\frac12
$$

时都经过：

$$
x=5
$$

但速度分别为：

$$
+10,\qquad-10
$$

因此，同一个位置可能对应冲突的速度监督。

如果使用满足适当条件的 OT Monge Map：

$$
X_1=T(X_0)
$$

并令：

$$
\sigma=0
$$

则：

$$
X_t=(1-t)X_0+tT(X_0)
$$

在位移插值映射可逆的情况下：

$$
X_0=T_t^{-1}(X_t)
$$

于是：

$$
U=T(X_0)-X_0
$$

能够由 $X_t$ 唯一确定。

因此：

$$
\boxed{
\operatorname{Var}(U\mid X_t)=0
}
\tag{E.2}
$$

这说明 OT Coupling 可以在适当条件下降低监督目标的随机性。

但对于有限平滑噪声：

$$
\sigma>0
$$

或有限 Batch 的近似 OT 耦合，该条件方差一般不会严格为零。

这也是论文将 OT-CFM 更快训练解释为监督方差降低时，需要保留的理论限定。

---

# 附录 F：Variance Reduction by Averaging Across Batches

原论文 Appendix C.1 还讨论了一种降低监督方差的方法。

设多个条件变量：

$$
\bar z=(z_1,\ldots,z_m)
$$

构造混合条件概率路径：

$$
\boxed{
p_t(x\mid\bar z)
=
\frac1m\sum_{i=1}^mp_t(x\mid z_i)
}
\tag{F.1}
$$

对应的混合速度场为：

$$
\boxed{
u_t(x\mid\bar z)
=
\frac{
\sum_{i=1}^m
p_t(x\mid z_i)u_t(x\mid z_i)
}{
\sum_{i=1}^m p_t(x\mid z_i)
}
}
\tag{F.2}
$$

这里需要注意，速度不是简单的算术平均。

因为在给定位置 $x$ 后，不同条件路径产生该位置的概率不同。

因此权重与：

$$
p_t(x\mid z_i)
$$

有关。

随着混合样本数增加，条件速度监督能够利用更多信息，从而减少单个条件样本带来的随机性。

但计算：

$$
u_t(x\mid\bar z)
$$

需要同时考虑多个条件路径，因此会增加计算成本。

所以存在：

$$
\boxed{
\text{监督方差降低}
\quad\longleftrightarrow\quad
\text{计算成本增加}
}
$$

的权衡。

---

# 附录 G：Energy-Based CFM

原论文 Appendix C.2 讨论了无法直接从目标分布采样、但可以计算未归一化密度的情况。

假设：

$$
R_0(x),\qquad R_1(x)
$$

是两个未归一化密度。

真实分布为：

$$
q_0(x)=\frac{R_0(x)}{Z_0}
$$

$$
q_1(x)=\frac{R_1(x)}{Z_1}
$$

其中：

$$
Z_0=\int R_0(x)\,dx
$$

$$
Z_1=\int R_1(x)\,dx
$$

是未知归一化常数。

假设可以从 Proposal Distribution：

$$
\widehat q_0,\qquad\widehat q_1
$$

采样。

定义重要性权重：

$$
\boxed{
w(x_0,x_1)
=
\frac{
R_0(x_0)R_1(x_1)
}{
\widehat q_0(x_0)\widehat q_1(x_1)
}
}
\tag{G.1}
$$

于是可以构造加权 CFM 损失：

$$
\boxed{
\mathcal L_{\mathrm{ECFM}}
=
\mathbb E_{\widehat q_0,\widehat q_1,t,X_t}
\left[
w(X_0,X_1)
\|v_\theta(t,X_t)-u_t(X_t\mid X_0,X_1)\|^2
\right]
}
\tag{G.2}
$$

由于：

$$
R_0=Z_0q_0
$$

$$
R_1=Z_1q_1
$$

利用重要性采样恒等式，可以得到：

$$
\boxed{
\mathcal L_{\mathrm{ECFM}}
=
Z_0Z_1\mathcal L_{\mathrm{I-CFM}}
}
\tag{G.3}
$$

其中等式右侧对应同一条件路径构造下、从真实 $q_0,q_1$ 采样的 I-CFM 目标。

因为：

$$
Z_0Z_1
$$

与模型参数无关且为正，所以两种损失具有相同的最优解。

因此，即使无法直接从真实目标分布采样，也可以通过重要性加权构造 CFM 训练目标。

不过，重要性权重可能具有较大方差，尤其是在高维空间中，这会带来训练困难。

---

# 全文核心公式总结

| 数学对象 | 公式 | 主要作用 |
|---|---|---|
| Neural ODE | $\dot X_t=v_\theta(t,X_t)$ | 定义连续时间生成动力学 |
| Gaussian Probability Flow | $u_t=\dot\mu_t+\frac{\dot\sigma_t}{\sigma_t}(x-\mu_t)$ | 构造可计算的条件速度 |
| Static OT | $\min_\pi\mathbb E_\pi\|X_1-X_0\|^2$ | 优化端点配对 |
| Dynamic OT | $\min_{p,v}\int_0^1\mathbb E_{p_t}\|v_t\|^2dt$ | 优化整个运输过程 |
| Benamou–Brenier | $\min_\pi\mathcal C(\pi)=\min_{p,v}\mathcal A(p,v)$ | 建立静态 OT 与动态 OT 的联系 |
| I-CFM | $X_t=(1-t)X_0+tX_1+\sigma Z$ | 处理一般源分布 |
| I-CFM 速度 | $U=X_1-X_0$ | 简单的条件速度监督 |
| OT-CFM | $(X_0,X_1)\sim\pi^*$ | 通过 OT 配对改善运输性质 |
| Minibatch OT | $\min_P\sum_{ij}P_{ij}C_{ij}$ | 在有限 Batch 中近似 OT |
| Entropic OT | $\min_\pi\mathbb E_\pi[c]+\varepsilon D_{\mathrm{KL}}(\pi\|q_0\otimes q_1)$ | 在运输成本与耦合随机性之间权衡 |
| Brownian Bridge | $X_t=(1-t)X_0+tX_1+\sigma\sqrt{t(1-t)}Z$ | 构造固定端点的条件概率路径 |
| SB-CFM | $(X_0,X_1)\sim\pi_{2\sigma^2}$ | 恢复 Schrödinger Bridge 的边缘概率流 |
| CFM 损失 | $\mathbb E\|v_\theta-u_t(\cdot\mid z)\|^2$ | 不模拟 ODE，直接训练速度网络 |

---

# 最终总结

这篇论文最核心的思想是：

**CFM 将连续时间生成模型的训练，转化为对已知条件概率路径的速度回归；而本文进一步表明，端点 Coupling 的选择能够决定概率流的重要运输性质。**

其中，三个层次需要特别区分。

**第一层：条件路径。**

给定一对端点：

$$
(X_0,X_1)
$$

构造：

$$
X_t=(1-t)X_0+tX_1
$$

得到的是一条匀速直线条件路径。

**第二层：边缘概率路径。**

对端点 Coupling：

$$
\pi(x_0,x_1)
$$

进行混合，得到：

$$
p_t(x)
$$

不同 Coupling 会产生不同的边缘概率路径。

如果使用真实 OT 耦合，在零平滑极限下能够得到动态最优传输路径。

**第三层：神经网络学习。**

CFM 使用条件速度监督：

$$
u_t(x\mid z)
$$

训练：

$$
v_\theta(t,x)
$$

使其近似生成目标边缘概率路径的速度场。

因此，OT-CFM 的理论逻辑是：

$$
\boxed{
\begin{aligned}
&\text{Static OT：优化端点配对}\\
&\Downarrow\\
&\text{Linear Interpolation：构造位移插值}\\
&\Downarrow\\
&\text{Dynamic OT：获得理论最优概率路径}\\
&\Downarrow\\
&\text{CFM：学习该路径对应的速度场}
\end{aligned}
}
$$

而 SB-CFM 将这个思想推广到随机运输：

$$
\boxed{
\begin{aligned}
&\text{Entropic OT：确定随机端点耦合}\\
&\Downarrow\\
&\text{Brownian Bridge：确定条件随机路径}\\
&\Downarrow\\
&\text{Schrödinger Bridge：得到最优随机过程的边缘分布}\\
&\Downarrow\\
&\text{CFM：学习相同边缘概率路径的确定性 ODE}
\end{aligned}
}
$$

这两个构造是全文最重要的理论结果。

**最终，本文的贡献不只是提出一种新的 Flow Matching 损失，而是建立了一个统一框架：通过设计静态端点 Coupling 和条件概率路径，在无需训练期动力学模拟的情况下，学习具有特定运输性质的连续时间生成模型。**

**原论文：** [Improving and Generalizing Flow-Based Generative Models with Minibatch Optimal Transport](https://arxiv.org/abs/2302.00482)